## Test the heuristic layout design strategy

In [1]:
import os
import sys
sys.path.append("..")
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *

result_dir = f"result/layout/heuristic/example_{get_timestamp()}"
os.makedirs(result_dir, exist_ok=True)

package = Package((100, 100), FabricateModel(compute_geometry="hexagon"), design_strategy="heuristic", result_dir=result_dir)

package.save_design_history()
package.to_pickle()
print("Package score:", package.estimate_score())

Package score: 3.3718061042940417


## Test the greedy layout design strategy

In [2]:
import os
import sys
sys.path.append("..")
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *

result_dir = f"result/layout/greedy/example_{get_timestamp()}"
os.makedirs(result_dir, exist_ok=True)

package = Package((50, 50), FabricateModel(20, 10), design_strategy="greedy", result_dir=result_dir)
# package = Package(design_strategy="greedy", result_dir=result_dir)

package.save_design_history()
package.to_pickle()
print("Package score:", package.estimate_score())

Package score: 16.954895465401776


In [5]:
import os
import sys
sys.path.append("..")
import logging
logging.basicConfig(level=logging.INFO)

from qunoc.element.package import *

result_dir = f"../evaluate/result/layout_scalability/package_size/100_100_greedy"
package = Package.from_pickle(filedir=result_dir)
print("Package score:", package.estimate_score())

Package score: 8.315169487561969


## Test the rl-based layout design strategy

In [1]:
import os
import sys
sys.path.append("..")
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from qunoc.layout.env import QuNoCEnv
from qunoc.layout.agent import LayoutAgent

result_dir = f"result/layout/rl/example_{get_timestamp()}"
os.makedirs(result_dir, exist_ok=True)

env = QuNoCEnv((40, 40), FabricateModel(), result_dir=result_dir)
# env = QuNoCEnv(result_dir=result_dir)
agent = LayoutAgent(env, result_dir=result_dir)
agent.train(1000)
package = agent.design()

package.save_design_history()
package.to_pickle()
print("Package score:", package.estimate_score())

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Training agent...
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 14        |
|    ep_rew_mean     | -1.22e+09 |
| time/              |           |
|    fps             | 1         |
|    iterations      | 1         |
|    time_elapsed    | 1315      |
|    total_timesteps | 2048      |
----------------------------------
Designing layout...
Initial score: 4.5200184374080035
New action: [25 16  1  1  0]
New score: 12.662205799533119
New action: [25 29  1  1  0]
New score: 14.201020764881791
New action: [25 29  1  1  0]
New score: 13.336246916800256
New action: [25 29  1  1  0]
New score: 13.321073056324037
New action: [25 29  1  1  0]
New score: 13.215733910589798
New action: [25 29  1  1  0]
New score: 13.28744778413076
New action: [25 29  1  1  0]
New score: 12.605992069741248
New action: [25 29  1  1  0]
New score: 12.634506247228455
New action: 

## Test the benchmark generation

In [ ]:
import os
import sys
sys.path.append("..")
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.element.benchmark import *

vqe_program = vqe(5)
print(vqe_program)
qft_program = qft(5)
print(qft_program)
qaoa_program = qaoa(5)
print(qaoa_program)
bv_program = bv(5)
print(bv_program)

vqe program: Control_0_1, Control_0_2, Control_0_3, Control_0_4, Control_1_2, Control_1_3, Control_1_4, Control_2_3, Control_2_4, Control_3_4
qft program: Control_1_0, Control_2_0, Control_3_0, Control_2_1, Control_4_0, Control_3_1, Control_4_1, Control_3_2, Control_4_2, Control_4_3
qaoa program: Control_2_4, Control_3_4, Control_0_2, Control_1_4, Control_1_2
bv program: Control_0_3, Control_0_4


## Test the routing algorithm

In [ ]:
import os
import sys
sys.path.append("..")

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from qunoc.element.benchmark import *

algo = "qft"
data_qubit_num = 16

# result_dir = f"result/routing/example_{get_timestamp()}"
result_dir = f"result/routing/example_{algo}_{data_qubit_num}"
os.makedirs(result_dir, exist_ok=True)

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    filename=os.path.join(result_dir, "log.txt"),
    filemode='w',
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO,
    encoding="utf-8"
)

package = Package(package_size=(9, 9), fab_model=FabricateModel(9, 9), design_strategy="manual", result_dir=result_dir)
package.design_module_top_down(package.compute_modules[0], DesignTemplate.Grid, DesignOrientation.X, 3)

package.save_design_history()
package.to_pickle()
print("Package score:", package.estimate_score())

program = get_program(algo, data_qubit_num)
gate_depend_fig = program.draw_depend()
gate_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_gate_depend.png"), bbox_inches="tight")
qiskit_circuit_fig = program.draw_circuit()
qiskit_circuit_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qiskit_circuit.png"), bbox_inches="tight")
print("Program gate num:", program.gate_depend_graph.number_of_nodes())

Package score: 43.05567281103211
Program gate num: 121


In [ ]:
from qunoc.utils.config import config
from qunoc.routing.router import *

config.debug_mapping = True
config.debug_block = True
config.debug_action = True
router = Router(package, program, result_dir=result_dir)
router.map_qubits()
print(f"Routing program {program.algo_name}-{program.data_qubit_num}")

Routing program qft-16


In [ ]:
router.compile_blocks_and_actions()
router.save_block_history()
block_depend_fig = router.draw_block_depend()
block_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_block_depend.png"), bbox_inches="tight")
print("Block depth:", router.get_block_depth(), ", block count:", router.block_depend_graph.number_of_nodes())

Block depth: 22 , block count: 59


In [ ]:
try:
    router.schedule_actions_to_circuit()
except AssertionError:
    router.save_action_history()
    raise
router.save_action_history()
action_depend_fig = router.draw_action_depend()
action_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_action_depend.png"), bbox_inches="tight")
print("Action depth:", router.get_action_depth(), ", action count:", router.action_depend_graph.number_of_nodes())
circuit = router.circuit
circuit.to_pickle(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qunoc_circuit.pkl"))
print("QuNoC gate fidelity:", router.estimate_qunoc_logic_gate_fide(), ", QuNoC circuit depth:", router.estimate_qunoc_circuit_duration())

Action depth: 353 , action count: 885
QuNoC gate fidelity: 0.8413625755790922 , QuNoC circuit depth: 29123.785842495003


## Test the immediate-schedule routing

In [1]:
import os
import sys
sys.path.append("..")

from qunoc.element.package import *
from qunoc.element.benchmark import *

algo = "vqe"
data_qubit_num = 100

result_dir = f"result/routing/example_{algo}_{data_qubit_num}"
os.makedirs(result_dir, exist_ok=True)

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    filename=os.path.join(result_dir, "log.txt"),
    filemode='w',
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO,
    encoding="utf-8"
)

package = Package.from_pickle(filepath=os.path.join("../evaluate/result/layout_scalability/package_size", f"40_40_greedy/package.pkl"))

program = get_program(algo, data_qubit_num)
gate_depend_fig = program.draw_depend()
gate_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_gate_depend.png"), bbox_inches="tight")
qiskit_circuit_fig = program.draw_circuit()
qiskit_circuit_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qiskit_circuit.png"), bbox_inches="tight")
print("Program gate num:", program.gate_depend_graph.number_of_nodes())

Program gate num: 617


In [2]:
from qunoc.utils.config import config
from qunoc.routing.router import *

config.debug_mapping = True
config.debug_block = True
config.debug_action = True
router = Router(package, program, result_dir=result_dir)
router.map_qubits()
print(f"Routing program {program.algo_name}-{program.data_qubit_num}")

Routing program vqe-100


In [3]:
try:
    router.compile_blocks_and_schedule_actions()
except AssertionError:
    router.save_action_history()
    raise
router.save_block_history()
# block_depend_fig = router.draw_block_depend()
# block_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_block_depend.png"), bbox_inches="tight")
# print("Block depth:", router.get_block_depth(), ", block count:", router.block_depend_graph.number_of_nodes())
router.save_action_history()
# action_depend_fig = router.draw_action_depend()
# action_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_action_depend.png"), bbox_inches="tight")
# print("Action depth:", router.get_action_depth(), ", action count:", router.action_depend_graph.number_of_nodes())
circuit = router.circuit
circuit.to_pickle(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qunoc_circuit.pkl"))
print("QuNoC gate fidelity:", router.estimate_qunoc_logic_gate_fide(), ", QuNoC circuit depth:", router.estimate_qunoc_circuit_duration())

QuNoC gate fidelity: 0.8481966116712604 , QuNoC circuit depth: 775492.0


In [4]:
try:
    router.dummy_compile_blocks_and_schedule_actions()
except AssertionError:
    # router.save_action_history()
    raise
# router.save_block_history()
# block_depend_fig = router.draw_block_depend()
# block_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_block_depend.png"), bbox_inches="tight")
# print("Block depth:", router.get_block_depth(), ", block count:", router.block_depend_graph.number_of_nodes())
# router.save_action_history()
# action_depend_fig = router.draw_action_depend()
# action_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_action_depend.png"), bbox_inches="tight")
# print("Action depth:", router.get_action_depth(), ", action count:", router.action_depend_graph.number_of_nodes())
dummy_circuit = router.dummy_circuit
dummy_circuit.to_pickle(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_dummy_circuit.pkl"))
print("Dummy gate fidelity:", router.estimate_dummy_logic_gate_fide(), ", Dummy circuit depth:", router.estimate_dummy_circuit_duration())

Dummy gate fidelity: 0.8481966116712604 , Dummy circuit depth: 904653.0


## Test the baseline MECH

In [ ]:
import os
import sys
sys.path.append("..")
import json
import math
%load_ext autoreload
%autoreload 2

from baseline.MECH.MECHBenchmarks import *
from baseline.mech_utils import get_mech_circuit_duration, get_mech_logic_gate_fide

structure="square"
chiplet_array_dim=(4,4)
chiplet_size=(10,10)
benchmarks=["s_ghz"]
data_qubit_num = 100

exp_results, circuits = MECH_experiments(structure=structure, chiplet_array_dim=chiplet_array_dim, chiplet_size=chiplet_size, benchmarks=benchmarks, data_qubit_num=data_qubit_num, cross_link_sparsity=1)
for b in benchmarks:
    mech_gate_fidelity = get_mech_logic_gate_fide(exp_results[b], chiplet_size=chiplet_size)
    mech_circuit_depth = get_mech_circuit_duration(exp_results[b])
    print("Benchmark:", b)
    print("Experiment result:", exp_results[b])
    print("MECH gate fidelity:", mech_gate_fidelity)
    print("MECH circuit depth:", mech_circuit_depth)

In [1]:
import os
import sys
sys.path.append("..")
import json
import math
%load_ext autoreload
%autoreload 2

from baseline.MECH.MECHBenchmarks import *

program = CIRCULAR(16)
for control_block in program:
    print(control_block.control, control_block.target_counter)

0 Counter({1: 1})
1 Counter({2: 1})
2 Counter({3: 1})
3 Counter({4: 1})
4 Counter({5: 1})
5 Counter({6: 1})
6 Counter({7: 1})
7 Counter({8: 1})
8 Counter({9: 1})
9 Counter({10: 1})
10 Counter({11: 1})
11 Counter({12: 1})
12 Counter({13: 1})
13 Counter({14: 1})
14 Counter({15: 1})
15 Counter({0: 1})


## Debug evaluation

In [1]:
import os
import sys
sys.path.append("..")
import json

from qunoc.element.package import *
from qunoc.element.benchmark import *
from qunoc.routing.router import *
from qunoc.utils.config import config
from baseline.MECH.MECHBenchmarks import MECH_experiments
from baseline.mech_utils import get_mech_circuit_duration, get_mech_logic_gate_fide

algorithm = "vqe"
package_length = 60
geometry = "square"

result_dir = f"result/routing/example_{algorithm}_{package_length}_{package_length}"
os.makedirs(result_dir, exist_ok=True)

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    filename=os.path.join(result_dir, "log.txt"),
    filemode='w',
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO,
    encoding="utf-8"
)

config.debug_mapping = True
config.debug_block = True
config.debug_action = True

for baseline in ["mech", "qunoc"]:
    package_size = (package_length, package_length)
    node_num = math.prod(package_size)
    data_qubit_num = int(100 * 2 ** math.log(node_num / 1600, 5))

    if baseline == "mech":
        if package_length < 100:
            chiplet_array_dim = (package_length // 10, package_length // 10)
            chiplet_size = (10, 10)
        else:
            chiplet_array_dim = (package_length // 20, package_length // 20)
            chiplet_size = (20, 20)
        cross_link_sparsity = chiplet_size[0] // FabricateModel().inter_sparsity
        exp_result_dict, mech_circuit = MECH_experiments(structure="square", chiplet_array_dim=chiplet_array_dim, chiplet_size=chiplet_size, benchmarks=[algorithm], data_qubit_num=data_qubit_num, cross_link_sparsity=cross_link_sparsity)
        with open(os.path.join(result_dir, "mech_circuit.pkl"), "wb") as f:
            pickle.dump(mech_circuit, f)
        exp_result = exp_result_dict[algorithm]
        circuit_duration = get_mech_circuit_duration(exp_result)
        logic_gate_fide = get_mech_logic_gate_fide(exp_result, (10, 10))
    elif baseline == "qunoc":
        package = Package.from_pickle(filepath=os.path.join("../evaluate/result/layout_scalability/geometry", f"{geometry}_greedy/package.pkl"))
        program = get_program(algorithm, data_qubit_num)
        router = Router(package, program, result_dir=result_dir)
        circuit = router.route_program_on_package()
        circuit.to_pickle(os.path.join(result_dir, f"qunoc_circuit.pkl"))
        intra_gate_num, inter_gate_num, qst_dist, meas_num, total_fide = router.statistic_route_results()
        circuit_duration = router.estimate_qunoc_circuit_duration()
        logic_gate_fide = router.estimate_qunoc_logic_gate_fide()
        exp_result = {
            "data_qubit_num": data_qubit_num,
            "logic_gate_num": program.get_gate_depend_graph().number_of_nodes() - 1, 
            "duration": circuit_duration,
            "fidelity": logic_gate_fide,
            "intra_gate": intra_gate_num,
            "inter_gate": inter_gate_num, 
            "qst_dist": qst_dist, 
            "meas_num": meas_num, 
            "block_num": router.get_block_num(),
            "sparsity": FabricateModel().inter_sparsity, 
        }
    
    with open(os.path.join(result_dir, f"{baseline}_exp_result.json"), 'w') as json_file:
        json.dump(exp_result, json_file)
    print(f"{baseline} experiment result:", exp_result)
    print(f"{baseline} circuit duration:", circuit_duration, ", logic gate fidelity:", logic_gate_fide)

848
mech experiment result: {'data_qubit_num': 141, 'logic_gate_num': 848, 'depth': 16942, 'eff_gate_num': 153766.0, 'prep_exec_depth': 15162, 'meas_depth': 842, 'on_chip': 58584, 'cross_chip': 3483, 'meas_num': 31549, 'weight': (7.4, 2.2), 'periods': (17, 2), 'shuttle_num': 842, 'sparsity': 3}
mech circuit duration: 38627760 , logic gate fidelity: 0.4155991502904192
qunoc experiment result: {'data_qubit_num': 141, 'logic_gate_num': 828, 'duration': 835772.0, 'fidelity': 0.7452959296548028, 'intra_gate': 2716, 'inter_gate': 4580, 'qst_dist': 84144, 'meas_num': 1656, 'block_num': 828, 'sparsity': 3}
qunoc circuit duration: 835772.0 , logic gate fidelity: 0.7452959296548028


## Compare greedy and heuristic strategies

In [1]:
import os
import sys
sys.path.append("..")
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from baseline.mech_utils import get_mech_highway_proportion

package_size = (100, 100)
fab_model = FabricateModel()
highway_proportion = get_mech_highway_proportion(
    structure=fab_model.compute_geometry,
    chiplet_array_dim=(fab_model.max_square_size, fab_model.max_square_size),
    chiplet_size=(package_size[0] // fab_model.max_square_size, package_size[1] // fab_model.max_square_size),
    cross_link_sparsity=fab_model.inter_sparsity,
)
print("MECH compute ratio:", 1 - highway_proportion)
for design_strategy in ["baseline", "heuristic", "brute", "greedy"]:
    result_dir = f"result/layout/{design_strategy}/example_{get_timestamp()}"
    os.makedirs(result_dir, exist_ok=True)
    package = Package(package_size, fab_model, compute_ratio=1-highway_proportion, design_strategy=design_strategy, result_dir=result_dir)
    package.save_design_history()
    package.to_pickle()
    score = package.estimate_score()
    fidelity = package.estimate_transfer()[0]
    compute_proportion = len(package.compute_nodes) / (package.package_size[0] * package.package_size[1])
    normalized_fide = fidelity * compute_proportion
    print(f"{design_strategy} package score:", score, "fidelity:", fidelity, "compute proportion:", compute_proportion, "normalized fidelity:", normalized_fide)

baseline package score: 0.8014975445292729 fidelity: 0.31902421647199086 compute proportion: 0.9604 normalized fidelity: 0.3063908574997


AssertionError: Invalid design: module_ComputeModule_67_1_98_65, template_DesignTemplate.Grid, orient_DesignOrientation.Y, interval_20

In [2]:
# from qunoc.element.chiplet import FabricateModel

# fab_model = FabricateModel()

# print(fab_model.get_intra_fide(100))

# print((fab_model.get_intra_fide(100) ** 8) * (0.983 ** 2))

print(125 / (5 ** 0.5))

55.90169943749474


## Test duration

In [1]:
import os
import sys
sys.path.append("..")
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from qunoc.routing.circuit import Circuit

algorithm_list = ["qft", "qaoa", "vqe"]
package_size = list(range(60, 140, 20))

intra_num_mech = []
inter_num_mech = []
intra_num_qunoc = []
inter_num_qunoc = []
qst_dist_qunoc = []

for (algorithm, size) in product(algorithm_list, package_size):
    circuit = Circuit.from_pickle(f"../evaluate/result/routing_scalability/overall/{size}_{size}_{algorithm}_qunoc/qunoc_circuit.pkl")
    package = Package.from_pickle(filepath=os.path.join("../evaluate/result/layout_scalability/package_size", f"{size}_{size}_greedy/package.pkl"))
    duration = circuit.estimate_duration(package)
    print(f"algorithm: {algorithm}, size: {size}, duration: {duration}")
    

algorithm: qft, size: 60, duration: 7754005.0
algorithm: qft, size: 80, duration: 20193011.0
algorithm: qft, size: 100, duration: 22696850.0
algorithm: qft, size: 120, duration: 29972353.0
algorithm: qaoa, size: 60, duration: 4288653.0
algorithm: qaoa, size: 80, duration: 10290178.0
algorithm: qaoa, size: 100, duration: 14580699.0
algorithm: qaoa, size: 120, duration: 21340153.0
algorithm: vqe, size: 60, duration: 1505873.0
algorithm: vqe, size: 80, duration: 2285654.0
algorithm: vqe, size: 100, duration: 3255087.0
algorithm: vqe, size: 120, duration: 4477843.0


## Test MECH with EQST

In [1]:
import os
import sys
sys.path.append("..")
import json
import math
%load_ext autoreload
%autoreload 2

from baseline.MECH.MECHBenchmarks import *
from baseline.mech_utils import get_mech_circuit_duration, get_mech_logic_gate_fide

structure="square"
chiplet_array_dim=(4,4)
chiplet_size=(10,10)
benchmarks=["qaoa"]
data_qubit_num = 16

exp_results, circuits = MECH_experiments(structure=structure, chiplet_array_dim=chiplet_array_dim, chiplet_size=chiplet_size, benchmarks=benchmarks, data_qubit_num=data_qubit_num, cross_link_sparsity=1)
for b in benchmarks:
    mech_gate_fidelity = get_mech_logic_gate_fide(exp_results[b], chiplet_size=chiplet_size)
    mech_circuit_depth = get_mech_circuit_duration(exp_results[b])
    print("Benchmark:", b)
    print("Experiment result:", exp_results[b])
    print("MECH gate fidelity:", mech_gate_fidelity)
    print("MECH circuit depth:", mech_circuit_depth)

exp_results, circuits = MECH_EQST_experiments(structure=structure, chiplet_array_dim=chiplet_array_dim, chiplet_size=chiplet_size, benchmarks=benchmarks, data_qubit_num=data_qubit_num, cross_link_sparsity=1)
for b in benchmarks:
    mech_gate_fidelity = get_mech_logic_gate_fide(exp_results[b], chiplet_size=chiplet_size)
    mech_circuit_depth = exp_results[b]["circuit_duration"]
    print("Benchmark:", b)
    print("Experiment result:", exp_results[b])
    print("MECH+EQST gate fidelity:", mech_gate_fidelity)
    print("MECH+EQST circuit depth:", mech_circuit_depth)

Benchmark: qaoa
Experiment result: {'data_qubit_num': 16, 'logic_gate_num': 60, 'depth': 366, 'eff_gate_num': 6587.0, 'prep_exec_depth': 324, 'meas_depth': 18, 'on_chip': 2478, 'cross_chip': 156, 'meas_num': 1343, 'weight': (7.4, 2.2), 'periods': (17, 2), 'shuttle_num': 18, 'sparsity': 1}
MECH gate fidelity: 0.5901484634407671
MECH circuit depth: 834480
Benchmark: qaoa
Experiment result: {'data_qubit_num': 16, 'logic_gate_num': 60, 'depth': 871, 'eff_gate_num': 1073.0, 'on_chip': 1073, 'cross_chip': 0, 'eqst_dist': 2916, 'meas_num': 0, 'weight': (7.4, 2.2), 'periods': (17, 2), 'shuttle_num': 0, 'sparsity': 1, 'circuit_duration': 871000}
MECH+EQST gate fidelity: 0.7504136711659174
MECH+EQST circuit depth: 871000


## Test dummy routing

In [1]:
import os
import sys
sys.path.append("..")

from qunoc.element.package import *
from qunoc.element.benchmark import *

algo = "vqe"
data_qubit_num = 16

result_dir = f"result/routing/example_{algo}_{data_qubit_num}"
os.makedirs(result_dir, exist_ok=True)

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    filename=os.path.join(result_dir, "log.txt"),
    filemode='w',
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO,
    encoding="utf-8"
)

package = Package(package_size=(9, 9), fab_model=FabricateModel(9, 9), design_strategy="manual", result_dir=result_dir)
package.design_module_top_down(package.compute_modules[0], DesignTemplate.Grid, DesignOrientation.X, 3)

package.save_design_history()
package.to_pickle()
print("Package score:", package.estimate_score())

program = get_program(algo, data_qubit_num)
gate_depend_fig = program.draw_depend()
gate_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_gate_depend.png"), bbox_inches="tight")
qiskit_circuit_fig = program.draw_circuit()
qiskit_circuit_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qiskit_circuit.png"), bbox_inches="tight")
print("Program gate num:", program.gate_depend_graph.number_of_nodes())

Package score: 33.04089604179082
Program gate num: 93


In [2]:
from qunoc.utils.config import config
from qunoc.routing.router import *

config.debug_mapping = True
config.debug_block = True
config.debug_action = True
router = Router(package, program, result_dir=result_dir)
router.map_qubits()
print(f"Routing program {program.algo_name}-{program.data_qubit_num}")

Routing program vqe-16


In [3]:
try:
    router.dummy_compile_blocks_and_schedule_actions()
except AssertionError:
    # router.save_action_history()
    raise
# router.save_block_history()
# block_depend_fig = router.draw_block_depend()
# block_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_block_depend.png"), bbox_inches="tight")
# print("Block depth:", router.get_block_depth(), ", block count:", router.block_depend_graph.number_of_nodes())
# router.save_action_history()
# action_depend_fig = router.draw_action_depend()
# action_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_action_depend.png"), bbox_inches="tight")
# print("Action depth:", router.get_action_depth(), ", action count:", router.action_depend_graph.number_of_nodes())
dummy_circuit = router.dummy_circuit
dummy_circuit.to_pickle(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_dummy_circuit.pkl"))
print("QuNoC gate fidelity:", router.estimate_dummy_logic_gate_fide(), ", QuNoC circuit depth:", router.estimate_dummy_circuit_duration())

QuNoC gate fidelity: 0.9230734644901324 , QuNoC circuit depth: 56929.0
